# Hallway Illuminance Train/Eval All-in-One

This notebook is the main user interface for the project. It is designed for Google Colab and walks through dataset preparation, manifest loading, dataloader creation, model setup, training, validation, testing, visualization, checkpointing, carbon estimation, ONNX export, and single-image inference.

## 1. Project Overview

The main task is hallway floor-plane illuminance estimation with a shared multi-task model. Public datasets act as auxiliary priors, while an optional custom hallway dataset provides the most direct supervision for lux maps, scalar lux values, and point-wise hallway reporting.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print(f'Project root: {PROJECT_ROOT}')

## 2. Dependency Installation

Run this once per Colab session.

In [ ]:
requirements_file = PROJECT_ROOT / 'requirements.txt'
print(f'Install dependencies from: {requirements_file}')
# In Colab, uncomment the next line:
# %pip install -q -r ../requirements.txt

## 3. Mounting Google Drive

Google Drive is the expected place for official dataset files, extracted folders, and persistent outputs.

In [ ]:
IN_COLAB = False
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    IN_COLAB = True
except ImportError:
    print('google.colab is not available in this environment.')

print(f'Running in Colab: {IN_COLAB}')

## 4. Loading Configs And Resolving Dataset Paths

Set each dataset input path to an extracted folder or supported archive. The custom hallway path can point to a folder containing a manifest CSV or directly to the CSV itself.

In [ ]:
import pandas as pd
import torch

from hallway_lighting.utils.io import ensure_dir, load_yaml

base_config = load_yaml(PROJECT_ROOT / 'configs' / 'base.yaml')
datasets_config = load_yaml(PROJECT_ROOT / 'configs' / 'datasets.yaml')
train_config = load_yaml(PROJECT_ROOT / 'configs' / 'train.yaml')
carbon_config = load_yaml(PROJECT_ROOT / 'configs' / 'carbon.yaml')
infer_config = load_yaml(PROJECT_ROOT / 'configs' / 'infer.yaml')

RUN_ROOT = ensure_dir(PROJECT_ROOT / 'runs' / 'notebook_run')
PREPARED_ROOT = ensure_dir(RUN_ROOT / 'prepared_datasets')
MANIFEST_DIR = ensure_dir(RUN_ROOT / 'manifests')
CHECKPOINT_DIR = ensure_dir(RUN_ROOT / 'checkpoints')
VIS_DIR = ensure_dir(RUN_ROOT / 'visualizations')
CONFIG_SNAPSHOT_DIR = ensure_dir(RUN_ROOT / 'config_snapshot')
HISTORY_PATH = RUN_ROOT / 'training_history.json'
BEST_MODEL_PATH = CHECKPOINT_DIR / 'best_model.pt'
LAST_MODEL_PATH = CHECKPOINT_DIR / 'last_model.pt'

DATASET_INPUTS = {
    'nyu_depth_v2': datasets_config['dataset_roots'].get('nyu_depth_v2', ''),
    'mit_intrinsic_images': datasets_config['dataset_roots'].get('mit_intrinsic_images', ''),
    'mid_intrinsics': datasets_config['dataset_roots'].get('mid_intrinsics', ''),
    'fast_sv_indoor_lighting': datasets_config['dataset_roots'].get('fast_sv_indoor_lighting', ''),
    'custom_hallway': datasets_config['dataset_roots'].get('custom_hallway', ''),
}

# Example Colab edits:
# DATASET_INPUTS['nyu_depth_v2'] = '/content/drive/MyDrive/datasets/nyu_depth_v2/nyu_depth_v2_labeled.mat'
# DATASET_INPUTS['mit_intrinsic_images'] = '/content/drive/MyDrive/datasets/mit_intrinsic_images'
# DATASET_INPUTS['mid_intrinsics'] = '/content/drive/MyDrive/datasets/mid_intrinsics'
# DATASET_INPUTS['fast_sv_indoor_lighting'] = '/content/drive/MyDrive/datasets/fast_sv_indoor_lighting'
# DATASET_INPUTS['custom_hallway'] = '/content/drive/MyDrive/datasets/custom_hallway/custom_hallway_manifest.csv'

display(pd.DataFrame({'dataset_name': list(DATASET_INPUTS.keys()), 'input_path': list(DATASET_INPUTS.values())}))
print(f'Run root: {RUN_ROOT}')

## 5. Extracting Archives And Building Manifests

The dataset registry prepares extracted folders and builds one normalized manifest per dataset. Missing labels are left empty and later skipped by the loss routing code.

In [ ]:
from hallway_lighting.data.dataset_registry import build_all_dataset_manifests

DATASET_RESULTS = build_all_dataset_manifests(
    dataset_inputs=DATASET_INPUTS,
    working_dir=PREPARED_ROOT,
    output_dir=MANIFEST_DIR,
    overwrite=False,
)

manifest_rows = []
for dataset_name, result in DATASET_RESULTS.items():
    manifest_rows.append(
        {
            'dataset_name': dataset_name,
            'prepared_root': str(result.prepared_input.prepared_root),
            'manifest_path': str(result.manifest_path),
            'rows': len(result.manifest),
        }
    )

display(pd.DataFrame(manifest_rows)) if manifest_rows else print('No dataset inputs were provided.')

## 6. Loading Manifests

This cell loads the saved manifest CSVs and shows split counts so you can confirm that `train`, `val`, and `test` rows exist before starting training.

In [ ]:
from hallway_lighting.data.manifests import load_manifest

MANIFESTS = {
    dataset_name: load_manifest(result.manifest_path)
    for dataset_name, result in DATASET_RESULTS.items()
}

if not MANIFESTS:
    print('No manifests are available yet.')
else:
    split_rows = []
    for dataset_name, manifest_df in MANIFESTS.items():
        counts = manifest_df['split'].fillna('unspecified').value_counts().to_dict()
        for split_name, count in counts.items():
            split_rows.append({'dataset_name': dataset_name, 'split': split_name, 'count': count})
        print(f'\n[{dataset_name}]')
        display(manifest_df.head())
    display(pd.DataFrame(split_rows))

### Manifest Coverage Diagnostics

This optional setup block summarizes how much supervision each dataset and split actually contributes. It is useful before training because the loss routing logic skips missing targets rather than inventing them.


In [ ]:
LABEL_COLUMNS = [
    'floor_mask_path',
    'lux_map_path',
    'avg_lux',
    'low_lux_p5',
    'high_lux_p95',
    'point_targets_json',
    'albedo_path',
    'reflectance_path',
    'shading_path',
    'gloss_path',
    'measured_power_w',
]

if not MANIFESTS:
    print('No manifests are available yet.')
else:
    coverage_rows = []
    for dataset_name, manifest_df in MANIFESTS.items():
        for split_name, split_df in manifest_df.groupby(manifest_df['split'].fillna('unspecified')):
            row = {
                'dataset_name': dataset_name,
                'split': split_name,
                'rows': len(split_df),
            }
            for column_name in LABEL_COLUMNS:
                if column_name not in split_df.columns:
                    row[column_name] = 0
                    continue
                values = split_df[column_name]
                if values.dtype == object:
                    row[column_name] = int(values.fillna('').astype(str).str.strip().ne('').sum())
                else:
                    row[column_name] = int(values.notna().sum())
            coverage_rows.append(row)
    coverage_df = pd.DataFrame(coverage_rows)
    display(coverage_df)


## 7. Creating Datasets And Dataloaders

The dataloader builder reads the normalized manifests, loads optional labels when they exist, and creates availability masks so losses can skip missing targets safely.

In [ ]:
from hallway_lighting.training import build_dataloaders
from hallway_lighting.utils.seed import set_seed

set_seed(base_config['project']['seed'])
IMAGE_SIZE = (
    train_config['training']['image_size']['height'],
    train_config['training']['image_size']['width'],
)
DATALOADERS = build_dataloaders(
    manifests=MANIFESTS,
    batch_size=train_config['training']['batch_size'],
    num_workers=train_config['training']['num_workers'],
    image_size=IMAGE_SIZE,
    seed=base_config['project']['seed'],
)

loader_summary = []
for split_name, loader in DATALOADERS.items():
    loader_summary.append(
        {
            'split': split_name,
            'available': loader is not None,
            'batches': 0 if loader is None else len(loader),
            'samples': 0 if loader is None else len(loader.dataset),
        }
    )
display(pd.DataFrame(loader_summary))

## 8. Initializing The Model

The model is instantiated from config, the optimizer and scheduler are created, mixed precision is enabled optionally, and checkpoint resume support is set up here.

In [ ]:
from hallway_lighting.models.hallway_multitask_unet import HallwayMultitaskUNet
from hallway_lighting.training import run_epoch
from hallway_lighting.utils.io import load_checkpoint, save_config_snapshot

device = torch.device('cuda' if torch.cuda.is_available() and train_config['training']['device'] == 'cuda' else 'cpu')
model = HallwayMultitaskUNet(base_config['model']).to(device)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=train_config['training']['learning_rate'],
    weight_decay=train_config['training']['weight_decay'],
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=max(1, train_config['training']['epochs']),
)
amp_enabled = bool(train_config['training']['amp']) and device.type == 'cuda'
scaler = torch.cuda.amp.GradScaler(enabled=amp_enabled)
save_config_snapshot(
    {
        'base': base_config,
        'datasets': datasets_config,
        'train': train_config,
        'carbon': carbon_config,
        'infer': infer_config,
    },
    CONFIG_SNAPSHOT_DIR,
)

HISTORY = {'train': [], 'val': [], 'test': []}
START_EPOCH = 0
BEST_VAL_SCORE = float('inf')
resume_path = train_config['training'].get('resume_checkpoint', '')
if resume_path:
    checkpoint = load_checkpoint(resume_path, model, optimizer=optimizer, scheduler=scheduler, scaler=scaler, map_location=device)
    HISTORY = checkpoint.get('history', HISTORY)
    START_EPOCH = int(checkpoint.get('epoch', -1)) + 1
    BEST_VAL_SCORE = float(checkpoint.get('extra_state', {}).get('best_val_score', BEST_VAL_SCORE))
    print(f'Resumed from checkpoint: {resume_path}')

dummy_input = torch.randn(1, 3, IMAGE_SIZE[0], IMAGE_SIZE[1], device=device)
with torch.no_grad():
    forward_output = model(dummy_input)
forward_rows = []
for key, value in forward_output.items():
    if isinstance(value, torch.Tensor):
        descriptor = tuple(value.shape)
    elif isinstance(value, dict):
        descriptor = list(value.keys())
    elif isinstance(value, list):
        descriptor = f'list[{len(value)}]'
    else:
        descriptor = str(type(value))
    forward_rows.append({'output_key': key, 'descriptor': descriptor})
display(pd.DataFrame(forward_rows))
print(f'Device: {device}')
print(f'AMP enabled: {amp_enabled}')

### Batch Inspection

This setup block pulls one batch from the train loader and shows what the notebook will actually feed into the model: dataset mix, tensor shapes, and how many samples in the batch have each optional target.


In [ ]:
TRAIN_BATCH_EXAMPLE = None
if DATALOADERS['train'] is None:
    print('Train loader is not available.')
else:
    TRAIN_BATCH_EXAMPLE = next(iter(DATALOADERS['train']))
    batch_summary_rows = [
        {'field': 'image', 'value': tuple(TRAIN_BATCH_EXAMPLE['image'].shape)},
        {'field': 'dataset_name', 'value': sorted(set(TRAIN_BATCH_EXAMPLE['dataset_name']))},
    ]
    for field_name in [
        'floor_mask', 'lux_map', 'albedo', 'gloss',
        'avg_lux', 'low_lux_p5', 'high_lux_p95', 'measured_power_w', 'interval_hours',
    ]:
        value = TRAIN_BATCH_EXAMPLE.get(field_name)
        mask = TRAIN_BATCH_EXAMPLE.get(f'{field_name}_valid_mask')
        batch_summary_rows.append({
            'field': field_name,
            'value': None if value is None else tuple(value.shape),
            'valid_count': None if mask is None else int(mask.sum().item()),
        })
    point_target_keys = [] if TRAIN_BATCH_EXAMPLE.get('point_lux') is None else sorted(TRAIN_BATCH_EXAMPLE['point_lux'].keys())
    batch_summary_rows.append({'field': 'point_lux_keys', 'value': point_target_keys})
    display(pd.DataFrame(batch_summary_rows))


### Model Summary And Checkpoint Helper

This setup block gives a compact model summary and defines a helper for loading the best or last checkpoint into the current notebook session.


In [ ]:
def count_parameters(module: torch.nn.Module) -> dict:
    total_params = sum(parameter.numel() for parameter in module.parameters())
    trainable_params = sum(parameter.numel() for parameter in module.parameters() if parameter.requires_grad)
    return {'total_parameters': total_params, 'trainable_parameters': trainable_params}

def load_named_checkpoint(checkpoint_name: str = 'best') -> None:
    global HISTORY, START_EPOCH, BEST_VAL_SCORE
    if checkpoint_name == 'best':
        checkpoint_path = BEST_MODEL_PATH
    elif checkpoint_name == 'last':
        checkpoint_path = LAST_MODEL_PATH
    else:
        raise ValueError("checkpoint_name must be 'best' or 'last'.")

    if not checkpoint_path.exists():
        raise FileNotFoundError(f'Checkpoint not found: {checkpoint_path}')

    checkpoint = load_checkpoint(checkpoint_path, model, optimizer=optimizer, scheduler=scheduler, scaler=scaler, map_location=device)
    HISTORY = checkpoint.get('history', HISTORY)
    START_EPOCH = int(checkpoint.get('epoch', -1)) + 1
    BEST_VAL_SCORE = float(checkpoint.get('extra_state', {}).get('best_val_score', BEST_VAL_SCORE))
    print(f'Loaded {checkpoint_name} checkpoint from {checkpoint_path}')

parameter_summary = count_parameters(model)
display(pd.DataFrame([parameter_summary]))


## 9. Training Loop

This is the main training cell. It runs epoch-level training, optional validation every epoch, checkpoint saving, and history logging.

In [ ]:
from hallway_lighting.utils.io import save_checkpoint, save_training_history

TRAIN_RESULT = None
VAL_RESULT = None

def run_training(num_epochs: int | None = None) -> dict:
    global HISTORY, START_EPOCH, BEST_VAL_SCORE, TRAIN_RESULT, VAL_RESULT

    train_loader = DATALOADERS['train']
    val_loader = DATALOADERS['val']
    if train_loader is None:
        raise ValueError('Training loader is not available. Check that your manifests contain train rows.')

    total_epochs = num_epochs or train_config['training']['epochs']
    for epoch in range(START_EPOCH, total_epochs):
        TRAIN_RESULT = run_epoch(
            model=model,
            dataloader=train_loader,
            device=device,
            loss_weights=train_config['loss_weights'],
            carbon_config=carbon_config,
            optimizer=optimizer,
            scaler=scaler,
            amp_enabled=amp_enabled,
            max_visualization_examples=train_config['training']['max_visualization_examples'],
            gradient_clip_norm=train_config['optimization']['gradient_clip_norm'],
        )
        train_summary = {'epoch': epoch + 1, **TRAIN_RESULT.summary}
        HISTORY['train'].append(train_summary)

        if val_loader is not None and ((epoch + 1) % train_config['training']['validate_every'] == 0):
            VAL_RESULT = run_epoch(
                model=model,
                dataloader=val_loader,
                device=device,
                loss_weights=train_config['loss_weights'],
                carbon_config=carbon_config,
                optimizer=None,
                scaler=None,
                amp_enabled=amp_enabled,
                max_visualization_examples=train_config['training']['max_visualization_examples'],
            )
            val_summary = {'epoch': epoch + 1, **VAL_RESULT.summary}
            HISTORY['val'].append(val_summary)
            current_val_score = VAL_RESULT.summary.get('avg_lux_error', VAL_RESULT.summary.get('total_loss', float('inf')))
            if current_val_score < BEST_VAL_SCORE:
                BEST_VAL_SCORE = current_val_score
                save_checkpoint(
                    model=model,
                    optimizer=optimizer,
                    scheduler=scheduler,
                    scaler=scaler,
                    epoch=epoch,
                    path=BEST_MODEL_PATH,
                    history=HISTORY,
                    extra_state={'best_val_score': BEST_VAL_SCORE},
                )

        scheduler.step()
        save_checkpoint(
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
            scaler=scaler,
            epoch=epoch,
            path=LAST_MODEL_PATH,
            history=HISTORY,
            extra_state={'best_val_score': BEST_VAL_SCORE},
        )
        save_training_history(HISTORY, HISTORY_PATH)

        print(
            f"Epoch {epoch + 1}/{total_epochs} | "
            f"train_total_loss={TRAIN_RESULT.summary.get('total_loss', 0.0):.4f} | "
            f"val_total_loss={0.0 if VAL_RESULT is None else VAL_RESULT.summary.get('total_loss', 0.0):.4f}"
        )
        START_EPOCH = epoch + 1

    return HISTORY

RUN_TRAINING = False
if RUN_TRAINING:
    run_training()
else:
    print('Set RUN_TRAINING = True to start training.')

## 10. Validation Loop

Use this cell to run validation on demand without entering the training loop again.

In [ ]:
def run_validation() -> dict:
    global VAL_RESULT
    if DATALOADERS['val'] is None:
        print('Validation loader is not available.')
        return {}

    VAL_RESULT = run_epoch(
        model=model,
        dataloader=DATALOADERS['val'],
        device=device,
        loss_weights=train_config['loss_weights'],
        carbon_config=carbon_config,
        optimizer=None,
        scaler=None,
        amp_enabled=amp_enabled,
        max_visualization_examples=train_config['training']['max_visualization_examples'],
    )
    display(pd.DataFrame([VAL_RESULT.summary]))
    return VAL_RESULT.summary

RUN_VALIDATION = False
if RUN_VALIDATION:
    run_validation()
else:
    print('Set RUN_VALIDATION = True to run validation.')

## 11. Testing / Evaluation Loop

This section runs the held-out test split and collects visual examples plus point-wise reports.

In [ ]:
TEST_RESULT = None

def run_test_evaluation() -> dict:
    global TEST_RESULT
    if DATALOADERS['test'] is None:
        print('Test loader is not available.')
        return {}

    TEST_RESULT = run_epoch(
        model=model,
        dataloader=DATALOADERS['test'],
        device=device,
        loss_weights=train_config['loss_weights'],
        carbon_config=carbon_config,
        optimizer=None,
        scaler=None,
        amp_enabled=amp_enabled,
        max_visualization_examples=train_config['training']['max_visualization_examples'],
    )
    display(pd.DataFrame([TEST_RESULT.summary]))
    if TEST_RESULT.point_reports:
        display(pd.DataFrame(TEST_RESULT.point_reports))
    return TEST_RESULT.summary

RUN_TEST = False
if RUN_TEST:
    run_test_evaluation()
else:
    print('Set RUN_TEST = True to run held-out evaluation.')

## 12. Visualizing Predictions

The helper utilities can show and save prediction panels with RGB input, lux heatmap, floor mask, optional albedo/gloss maps, and hallway point overlays.

In [ ]:
import matplotlib.pyplot as plt

from hallway_lighting.utils.visualization import create_prediction_figure, plot_pointwise_lux, save_prediction_figure

RESULT_FOR_VIS = TEST_RESULT if TEST_RESULT is not None else VAL_RESULT
if RESULT_FOR_VIS is None or not RESULT_FOR_VIS.visual_examples:
    print('Run validation or testing first to collect visual examples.')
else:
    for example_index, example in enumerate(RESULT_FOR_VIS.visual_examples[:train_config['training']['max_visualization_examples']]):
        figure = create_prediction_figure(
            image=example['image'],
            lux_map=example['lux_map_pred'],
            floor_mask_pred=example['floor_mask_pred'],
            albedo_pred=example['albedo_pred'],
            gloss_pred=example['gloss_pred'],
            points=example['point_targets'],
            point_values=example['point_lux_pred'],
            title=f"{example['dataset_name']} | {example['sample_id']}"
        )
        plt.show()
        save_prediction_figure(
            path=VIS_DIR / f"{example['sample_id']}_prediction.png",
            image=example['image'],
            lux_map=example['lux_map_pred'],
            floor_mask_pred=example['floor_mask_pred'],
            albedo_pred=example['albedo_pred'],
            gloss_pred=example['gloss_pred'],
            points=example['point_targets'],
            point_values=example['point_lux_pred'],
            title=f"{example['dataset_name']} | {example['sample_id']}"
        )
        plt.close(figure)
        if example['point_lux_pred']:
            plot_pointwise_lux(example['point_lux_pred'], title=f"Predicted point lux | {example['sample_id']}")

## 13. Saving Checkpoints

Training saves `best_model.pt`, `last_model.pt`, a JSON training history, and YAML config snapshots under the notebook run directory.

In [ ]:
checkpoint_rows = []
for checkpoint_path in sorted(CHECKPOINT_DIR.glob('*.pt')):
    checkpoint_rows.append({'checkpoint_file': checkpoint_path.name, 'path': str(checkpoint_path)})

display(pd.DataFrame(checkpoint_rows)) if checkpoint_rows else print('No checkpoints have been saved yet.')
print(f'History path: {HISTORY_PATH}')
print(f'Config snapshot dir: {CONFIG_SNAPSHOT_DIR}')
print(f'Visualization dir: {VIS_DIR}')

### History And Output Inspection

Use this setup block to inspect saved training history, config snapshots, and generated visualization files without leaving the notebook.


In [ ]:
import json
import matplotlib.pyplot as plt

if HISTORY_PATH.exists():
    history_payload = json.loads(HISTORY_PATH.read_text())
    train_history_df = pd.DataFrame(history_payload.get('train', []))
    val_history_df = pd.DataFrame(history_payload.get('val', []))
    test_history_df = pd.DataFrame(history_payload.get('test', []))

    if not train_history_df.empty:
        display(train_history_df.tail())
        plt.figure(figsize=(8, 4))
        plt.plot(train_history_df['epoch'], train_history_df['total_loss'], label='train_total_loss')
        if not val_history_df.empty and 'total_loss' in val_history_df:
            plt.plot(val_history_df['epoch'], val_history_df['total_loss'], label='val_total_loss')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.title('Training History')
        plt.legend()
        plt.tight_layout()
        plt.show()
    if not test_history_df.empty:
        display(test_history_df.tail())
else:
    print('No saved training history yet.')

output_summary = {
    'checkpoint_files': [path.name for path in sorted(CHECKPOINT_DIR.glob('*.pt'))],
    'visualization_files': [path.name for path in sorted(VIS_DIR.glob('*.png'))[:10]],
    'config_snapshot_files': [path.name for path in sorted(CONFIG_SNAPSHOT_DIR.glob('*.yaml'))],
}
output_summary


## 14. Point-wise Illuminance Reporting

Custom hallway samples can provide point targets such as `under_fixture_1` or `between_fixture_1_2`. Evaluation keeps predicted and target point reports for quick inspection.

In [ ]:
POINT_REPORT_SOURCE = TEST_RESULT if TEST_RESULT is not None else VAL_RESULT
if POINT_REPORT_SOURCE is None or not POINT_REPORT_SOURCE.point_reports:
    print('Run validation or testing on custom hallway data to populate point reports.')
else:
    point_report_df = pd.DataFrame(POINT_REPORT_SOURCE.point_reports)
    display(point_report_df)

## 15. Carbon Estimation

Carbon estimates are derived from predicted or measured power and interval duration. This section demonstrates the project-level reporting helper.

In [ ]:
from hallway_lighting.utils.carbon import summarize_carbon_from_lux

from hallway_lighting.utils.metrics import summarize_lux_map

avg_lux_for_carbon = 150.0
if TEST_RESULT is not None and TEST_RESULT.visual_examples:
    avg_lux_for_carbon = summarize_lux_map(TEST_RESULT.visual_examples[0]['lux_map_pred'])['avg_lux']

carbon_report = summarize_carbon_from_lux(
    avg_lux=avg_lux_for_carbon,
    floor_area_m2=infer_config['inference']['floor_area_m2'],
    watts_per_lux_m2=carbon_config['carbon']['watts_per_lux_m2'],
    interval_hours=carbon_config['carbon']['default_interval_hours'],
    carbon_factor_kg_per_kwh=carbon_config['carbon']['default_grid_carbon_factor_kg_per_kwh'],
)
carbon_report

## 16. ONNX Export

The multitask model returns dictionaries, so ONNX export uses a small wrapper that exposes tensor outputs only.

In [ ]:
class OnnxExportWrapper(torch.nn.Module):
    def __init__(self, wrapped_model: torch.nn.Module) -> None:
        super().__init__()
        self.wrapped_model = wrapped_model

    def forward(self, image: torch.Tensor):
        outputs = self.wrapped_model(image)
        return (
            outputs['lux_map'],
            outputs['avg_lux'],
            outputs['low_lux_p5'],
            outputs['high_lux_p95'],
            outputs['floor_mask_pred'],
            outputs['albedo_pred'],
            outputs['gloss_pred'],
            outputs['uncertainty_map'],
        )

onnx_path = PROJECT_ROOT / infer_config['inference']['export_onnx_path']
onnx_path.parent.mkdir(parents=True, exist_ok=True)
dummy_input = torch.randn(1, 3, IMAGE_SIZE[0], IMAGE_SIZE[1], device=device)

# Uncomment to export:
# torch.onnx.export(
#     OnnxExportWrapper(model).to(device),
#     dummy_input,
#     onnx_path,
#     input_names=['image'],
#     output_names=['lux_map', 'avg_lux', 'low_lux_p5', 'high_lux_p95', 'floor_mask_pred', 'albedo_pred', 'gloss_pred', 'uncertainty_map'],
#     opset_version=17,
# )
print(f'Planned ONNX export path: {onnx_path}')

## 17. Single-Image Inference

After training or loading a checkpoint, this section runs the single-image inference helper and reports lux, point-wise, power, and carbon outputs.

In [ ]:
from hallway_lighting.data.point_sampling import default_hallway_points
from hallway_lighting.infer import run_single_image_inference

single_image_path = ''
if single_image_path and Path(single_image_path).exists():
    inference_output = run_single_image_inference(
        model=model,
        image_path=single_image_path,
        device=str(device),
        image_size=IMAGE_SIZE,
        point_targets=default_hallway_points(base_config['model']['fixture_count']),
        floor_area_m2=infer_config['inference']['floor_area_m2'],
        watts_per_lux_m2=carbon_config['carbon']['watts_per_lux_m2'],
        carbon_factor_kg_per_kwh=carbon_config['carbon']['default_grid_carbon_factor_kg_per_kwh'],
        interval_hours=infer_config['inference']['interval_hours'],
    )
    print(inference_output)
else:
    print('Set single_image_path to a real hallway image before running inference.')

## 18. Next Steps

The notebook now contains the practical execution flow for manifest-driven training, validation, testing, visualization, and checkpointing. The main remaining work is improving dataset-to-head alignment for auxiliary public datasets and expanding the supervision coverage of the current multitask architecture.